# Practical Session 02d &mdash; Evaluation, polynomial features & overfitting

**Companion to Lecture&nbsp;02.** We can now *fit* linear regression two ways (closed form in 02b,
gradient descent in 02c). This notebook is about *judging* the fit and *stretching* the model:

* **score a regression honestly** &mdash; MSE, RMSE, MAE and $R^2$, implemented from scratch and
  checked against scikit-learn, always on **held-out** data;
* turn a straight line into a **curve** with **polynomial features** &mdash; still linear regression,
  just with more columns;
* watch **overfitting** appear as the degree grows, and preview the cure (**ridge**);
* finish on a **real** multivariate dataset.

> ⭐ **Key idea.** "Linear" refers to the *parameters*, not the shape of the curve. By feeding the
> model transformed features ($x^2, x^3, \dots$) we fit wiggly curves with the exact same
> least-squares machinery &mdash; and pay for the extra flexibility with **overfitting** unless we
> evaluate honestly and, when needed, regularize.

*Run each cell (Shift+Enter). Self-contained and offline (one optional cell uses a bundled real dataset).*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nb_utils import use_style
from nb_data import make_regression_1d, make_wave

use_style("notes")
np.set_printoptions(precision=3, suppress=True)
print("ready")

## 1. Scoring a regression, from scratch

Four standard numbers, each a one-liner:

* **MSE** = mean squared error $\tfrac1n\sum (y_i-\hat y_i)^2$ &mdash; the training objective.
* **RMSE** = $\sqrt{\text{MSE}}$ &mdash; same units as $y$, so it is interpretable.
* **MAE** = mean absolute error $\tfrac1n\sum |y_i-\hat y_i|$ &mdash; robust to outliers.
* **$R^2$** = $1 - \frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$ &mdash; the fraction of the target's
  variance the model explains; 1 is perfect, 0 is "no better than predicting the mean".

In [ ]:
def mse(y, yhat):  return np.mean((y - yhat) ** 2)
def rmse(y, yhat): return np.sqrt(mse(y, yhat))
def mae(y, yhat):  return np.mean(np.abs(y - yhat))
def r2(y, yhat):   return 1 - np.sum((y - yhat) ** 2) / np.sum((y - y.mean()) ** 2)

# check every one against scikit-learn on a quick fit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
x, y = make_regression_1d(n=60, seed=0)
X = np.column_stack([np.ones_like(x), x]); theta = np.linalg.lstsq(X, y, rcond=None)[0]
yhat = X @ theta

print(f"{'metric':6} {'ours':>10} {'sklearn':>10}")
print(f"{'MSE':6} {mse(y, yhat):10.4f} {mean_squared_error(y, yhat):10.4f}")
print(f"{'RMSE':6} {rmse(y, yhat):10.4f} {mean_squared_error(y, yhat)**0.5:10.4f}")
print(f"{'MAE':6} {mae(y, yhat):10.4f} {mean_absolute_error(y, yhat):10.4f}")
print(f"{'R2':6} {r2(y, yhat):10.4f} {r2_score(y, yhat):10.4f}")

## 2. Always score on held-out data

As notebook 01d hammered home: a model is judged on data it did **not** train on. Split, fit on
train, score on test, and look at the two diagnostic plots &mdash; predicted-vs-actual and residuals.

In [ ]:
from sklearn.model_selection import train_test_split
Xf, Xt, yf, yt = train_test_split(X, y, test_size=0.3, random_state=0)
theta = np.linalg.lstsq(Xf, yf, rcond=None)[0]
pred = Xt @ theta

fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.6, 3.4))
a1.scatter(yt, pred, s=22, alpha=0.7); lim = [yt.min(), yt.max()]
a1.plot(lim, lim, "k--", lw=1); a1.set_xlabel("actual y"); a1.set_ylabel("predicted y")
a1.set_title(f"predicted vs actual (test $R^2$={r2(yt, pred):.2f})")
a2.scatter(pred, yt - pred, s=22, alpha=0.7); a2.axhline(0, color="k", lw=1)
a2.set_xlabel("predicted y"); a2.set_ylabel("residual"); a2.set_title("residuals: structureless = good")
for a in (a1, a2): a.grid(alpha=0.3)
plt.show()
print(f"test  RMSE = {rmse(yt, pred):.3f} | MAE = {mae(yt, pred):.3f} | R2 = {r2(yt, pred):.3f}")

## 3. A caution: least squares chases outliers

Squared error punishes a big miss *quadratically*, so a single wild point can drag the whole line
toward it &mdash; the flip side of the Gaussian-noise assumption from 02b (real data has heavier
tails). Inject one outlier and watch the least-squares line tilt; then compare a **robust** fit
(Huber loss) that mostly ignores it.

In [ ]:
xo, yo = make_regression_1d(n=40, seed=0)
yo_out = yo.copy(); yo_out[6] += 22                # inject one big outlier

def fit_line(xv, yv):
    A = np.column_stack([np.ones_like(xv), xv]); return np.linalg.lstsq(A, yv, rcond=None)[0]

th_clean = fit_line(xo, yo)                         # least squares on the clean data
th_out   = fit_line(xo, yo_out)                     # least squares WITH the outlier

from sklearn.linear_model import HuberRegressor
hub = HuberRegressor().fit(xo.reshape(-1, 1), yo_out)   # robust: squared loss near 0, absolute in the tails

xg = np.linspace(xo.min(), xo.max(), 100)
fig, ax = plt.subplots(figsize=(6.6, 3.9))
ax.scatter(xo, yo_out, s=24, alpha=0.7, label="data (with 1 outlier)")
ax.scatter(xo[6], yo_out[6], s=110, facecolors="none", edgecolors="#D55E00", lw=2, label="the outlier")
ax.plot(xg, th_clean[0] + th_clean[1] * xg, "k:", label="least squares (no outlier)")
ax.plot(xg, th_out[0] + th_out[1] * xg, color="#D55E00", label="least squares (dragged)")
ax.plot(xg, hub.intercept_ + hub.coef_[0] * xg, color="#117733", lw=2, label="Huber (robust)")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_title("one outlier tilts the least-squares line")
ax.legend(fontsize=8)
plt.show()

> ⚠️ **Watch out.** The circled point pulls the orange least-squares line off the trend, while the
> green **Huber** line barely moves. If your data has outliers, either clean them (01c) or use a
> robust loss &mdash; and always *look* at a residual plot. Reporting **MAE** next to RMSE helps too: a
> large RMSE&ndash;MAE gap is the fingerprint of a few big errors.

## 4. From a line to a curve: polynomial features

A straight line cannot follow a curved target. The trick: invent new features that are **powers**
of $x$, so the design matrix becomes $[\,1,\ x,\ x^2,\ \dots,\ x^d\,]$. The model is still linear in
its parameters &mdash; we just fit it with the normal equations from 02b &mdash; but as a function of $x$
it is now a degree-$d$ polynomial.

In [ ]:
def poly_features(x, degree):
    return np.column_stack([x ** k for k in range(degree + 1)])   # [1, x, x^2, ..., x^d]

xw, yw = make_wave(n=30, seed=1)                # a wavy target: sin(2*pi*x) + noise
xs = np.linspace(0, 1, 200)

fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.scatter(xw, yw, s=28, alpha=0.7, zorder=3, label="data")
ax.plot(xs, np.sin(2*np.pi*xs), "k:", lw=1, label="true function")
for d, col in [(1, "#0072B2"), (3, "#009E73"), (15, "#D55E00")]:
    theta = np.linalg.lstsq(poly_features(xw, d), yw, rcond=None)[0]
    ax.plot(xs, poly_features(xs, d) @ theta, color=col, label=f"degree {d}")
ax.set_ylim(-1.8, 1.8); ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("degree 1 underfits, 3 fits well, 15 overfits"); ax.legend(fontsize=8)
plt.show()

Degree 1 (a line) is too rigid &mdash; it **underfits**. Degree 3 tracks the true curve nicely.
Degree 15 wiggles through every noisy point and swings wildly between them &mdash; it **overfits**.

### More than one input: interaction terms

Everything above used a **single** input $x$. With **several** inputs, "polynomial features" means
more than the powers of each feature on its own &mdash; it also includes **cross terms**
(*interactions*). For two inputs $x_1, x_2$ up to degree 2 the design matrix has columns

$$[\,1,\ \ x_1,\ \ x_2,\ \ x_1^2,\ \ \underbrace{x_1 x_2}_{\text{interaction}},\ \ x_2^2\,].$$

The interaction $x_1 x_2$ is the genuinely new ingredient: it lets the effect of one feature
**depend on the value of the other** &mdash; something a purely additive plane
$w_0+w_1x_1+w_2x_2$ can *never* represent. The price is a **combinatorial blow-up**: degree $p$ over
$d$ inputs has $\binom{d+p}{p}$ columns.

In [ ]:
from itertools import combinations_with_replacement
from math import comb

def poly_features_md(X, degree):
    """All monomials of X's columns up to total `degree` (including the bias 1).
    Builds the same columns as scikit-learn's sklearn.preprocessing.PolynomialFeatures."""
    X = np.atleast_2d(X)
    n, d = X.shape
    cols, names = [np.ones(n)], ["1"]
    for deg in range(1, degree + 1):
        for combo in combinations_with_replacement(range(d), deg):   # e.g. (0, 1) -> x1*x2
            cols.append(np.prod(X[:, combo], axis=1))
            names.append("*".join(f"x{j+1}" for j in combo))
    return np.column_stack(cols), names

# the 6 degree-2 features for two inputs -- note the x1*x2 interaction that a plane lacks:
_, names = poly_features_md(np.zeros((1, 2)), degree=2)
print("2 inputs, degree 2 ->", names)

# ...and how fast the column count grows -- C(d+p, p) -- the curse of dimensionality:
for d in (1, 2, 5, 10, 20):
    print(f"{d:2d} inputs, degree 3  ->  {comb(d + 3, 3):5d} features")

In [ ]:
from sklearn.model_selection import train_test_split

# a target with a PURE interaction: y = x1 * x2 (a saddle), plus a little noise
rng = np.random.default_rng(1)
Xmd = rng.uniform(-2, 2, size=(400, 2))
ymd = Xmd[:, 0] * Xmd[:, 1] + rng.normal(0, 0.4, 400)
Xtr, Xte, ytr, yte = train_test_split(Xmd, ymd, test_size=0.3, random_state=0)

# (a) additive linear model [1, x1, x2] -- no interaction column
th1   = np.linalg.lstsq(poly_features_md(Xtr, 1)[0], ytr, rcond=None)[0]
pred1 = poly_features_md(Xte, 1)[0] @ th1

# (b) degree-2 features -- now includes the x1*x2 cross term
Phi2, nm2 = poly_features_md(Xtr, 2)
th2   = np.linalg.lstsq(Phi2, ytr, rcond=None)[0]
pred2 = poly_features_md(Xte, 2)[0] @ th2

print(f"linear [1, x1, x2]    test R2 = {r2(yte, pred1):.3f}   (a flat plane -- hopeless here)")
print(f"degree 2 (+ x1*x2)    test R2 = {r2(yte, pred2):.3f}")
print("degree-2 weights:", {n: round(w, 2) for n, w in zip(nm2, th2)})   # x1*x2 ~ 1, the rest ~ 0

# see it: the true saddle, the flat linear plane, and the degree-2 fit
g = np.linspace(-2, 2, 60); G1, G2 = np.meshgrid(g, g)
grid = np.column_stack([G1.ravel(), G2.ravel()])
Z_true = G1 * G2
Z_lin  = (poly_features_md(grid, 1)[0] @ th1).reshape(G1.shape)
Z_deg2 = (poly_features_md(grid, 2)[0] @ th2).reshape(G1.shape)

vmax = np.abs(Z_true).max(); levels = np.linspace(-vmax, vmax, 15)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
titles = ["true  $y = x_1 x_2$",
          f"linear plane  ($R^2$={r2(yte, pred1):.2f})",
          f"degree 2 + interaction  ($R^2$={r2(yte, pred2):.2f})"]
for ax, Z, ttl in zip(axes, [Z_true, Z_lin, Z_deg2], titles):
    c = ax.contourf(G1, G2, Z, levels=levels, cmap="RdBu_r", vmin=-vmax, vmax=vmax, extend="both")
    ax.set_xlabel("$x_1$"); ax.set_title(ttl, fontsize=10)
axes[0].set_ylabel("$x_2$")
fig.colorbar(c, ax=axes, shrink=0.85, label="y")
plt.show()

> 💡 **Interactions are the multivariate ingredient.** The additive plane sits at $R^2\approx0$: the
> target $y=x_1x_2$ has no linear part, so $[1,x_1,x_2]$ can only fit a flat tilt. Adding the single
> **interaction** column $x_1x_2$ recovers the saddle (its weight comes out $\approx 1$, all others
> $\approx 0$) &mdash; this is how a linear model expresses "the effect of $x_1$ **depends on** $x_2$".
>
> The catch is the $\binom{d+p}{p}$ blow-up shown above &mdash; degree&nbsp;3 on 10 features is already
> 286 columns &mdash; so multivariate polynomials **overfit fast** and lean hard on the **ridge / lasso**
> regularization of Session&nbsp;3. (`sklearn.preprocessing.PolynomialFeatures` builds exactly these
> columns for you; `interaction_only=True` keeps just the cross terms.)

## 5. The overfitting U-curve, and exploding weights

Sweep the degree and score on both train and a held-out test set. Training error keeps dropping
(more flexibility fits the training points better), but test error bottoms out and then **rises** &mdash;
the bias&ndash;variance trade-off. A second tell-tale: the fitted **weights blow up**.

In [ ]:
xtr, ytr = make_wave(n=30, seed=1)
xte, yte = make_wave(n=200, seed=7)            # a big, fresh test set from the same process

degrees = range(0, 16)
tr_err, te_err, wmax = [], [], []
for d in degrees:
    th = np.linalg.lstsq(poly_features(xtr, d), ytr, rcond=None)[0]
    tr_err.append(rmse(ytr, poly_features(xtr, d) @ th))
    te_err.append(rmse(yte, poly_features(xte, d) @ th))
    wmax.append(np.abs(th).max())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.2, 3.6))
a1.plot(list(degrees), tr_err, "o-", label="train RMSE")
a1.plot(list(degrees), te_err, "s-", label="test RMSE")
best = int(np.argmin(te_err)); a1.axvline(best, color="k", ls=":", lw=1)
a1.set_ylim(0, 1.2)                                        # clip the exploding test error
a1.text(best + 0.3, 1.0, f"best degree = {best}", fontsize=9)   # inside the axes (avoids a huge canvas)
a1.set_xlabel("polynomial degree"); a1.set_ylabel("RMSE")
a1.set_title("train drops, test bottoms then rises"); a1.legend(fontsize=9); a1.grid(alpha=0.3)
a2.semilogy(list(degrees), wmax, "o-", color="#CC79A7")
a2.set_xlabel("polynomial degree"); a2.set_ylabel("largest |weight|  (log)")
a2.set_title("overfitting inflates the weights"); a2.grid(alpha=0.3)
plt.show()

## 6. The cure, previewed: ridge regression

Overfitting shows up as huge weights, so *charge a price for large weights*: add
$\lambda\lVert\boldsymbol\theta\rVert^2$ to the cost. The closed form barely changes &mdash; add
$\lambda\mathbf{I}$ before inverting:

$$\boldsymbol\theta_{\text{ridge}} = (\mathbf{X}^{\!\top}\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^{\!\top}\mathbf{y}.$$

Even a tiny $\lambda$ tames the wild degree-15 curve and shrinks the weights.

In [ ]:
def fit_ridge(X, y, lam):
    p = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)   # normal equations + lambda*I

Xp = poly_features(xw, 15)
th0  = np.linalg.lstsq(Xp, yw, rcond=None)[0]     # lambda = 0 (plain least squares)
thL  = fit_ridge(Xp, yw, lam=1e-3)                # a little ridge

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.scatter(xw, yw, s=28, alpha=0.7, zorder=3, label="data")
ax.plot(xs, np.sin(2*np.pi*xs), "k:", lw=1, label="true function")
ax.plot(xs, poly_features(xs, 15) @ th0, color="#D55E00", label="degree 15, no ridge")
ax.plot(xs, poly_features(xs, 15) @ thL, color="#117733", lw=2, label="degree 15, ridge $\\lambda$=1e-3")
ax.set_ylim(-1.8, 1.8); ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_title("ridge tames the wiggle")
ax.legend(fontsize=8)
plt.show()
print(f"largest |weight|:  no ridge = {np.abs(th0).max():,.0f}   with ridge = {np.abs(thL).max():.1f}")

> 🔬 **Going deeper.** Ridge (a.k.a. $L_2$ / Tikhonov regularization) also makes
> $\mathbf{X}^{\!\top}\mathbf{X}+\lambda\mathbf{I}$ invertible even when $\mathbf{X}^{\!\top}\mathbf{X}$
> is singular (the collinear-features problem from 02b). Choosing $\lambda$ well &mdash; and the sparse
> cousin **lasso** ($L_1$) &mdash; is the entire subject of **Session&nbsp;03**. Here it is just a
> one-line preview.

## 7. A real multivariate dataset

The lecture's running example is predicting **diabetes progression** from ten patient
measurements &mdash; a genuine, bundled scikit-learn dataset. We fit it with our from-scratch normal
equations, evaluate honestly, and read which measurements matter.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

data = load_diabetes()
Xd, yd, names = data.data, data.target, data.feature_names   # features are already standardized
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=0)

# from-scratch fit (add a bias column, solve the normal equations)
A_tr = np.column_stack([np.ones(len(Xtr)), Xtr])
theta = np.linalg.lstsq(A_tr, ytr, rcond=None)[0]
pred = np.column_stack([np.ones(len(Xte)), Xte]) @ theta

from sklearn.linear_model import LinearRegression
sk = LinearRegression().fit(Xtr, ytr)
print("our from-scratch fit matches sklearn:", np.allclose(theta[1:], sk.coef_) and np.isclose(theta[0], sk.intercept_))
print(f"\ntest RMSE = {rmse(yte, pred):.1f} | MAE = {mae(yte, pred):.1f} | R2 = {r2(yte, pred):.3f}")

In [ ]:
coef = theta[1:]
order = np.argsort(np.abs(coef))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.4, 3.6))
a1.barh([names[i] for i in order], coef[order], color="#0072B2")
a1.axvline(0, color="k", lw=0.8); a1.set_title("coefficients (features are standardized)"); a1.grid(alpha=0.3, axis="x")
a2.scatter(yte, pred, s=20, alpha=0.6); lim = [yte.min(), yte.max()]
a2.plot(lim, lim, "k--", lw=1); a2.set_xlabel("actual progression"); a2.set_ylabel("predicted")
a2.set_title(f"predicted vs actual (test $R^2$={r2(yte, pred):.2f})"); a2.grid(alpha=0.3)
plt.show()

> 💡 **Reading the coefficients.** Because the features were standardized, the coefficient
> magnitudes are directly comparable: **`bmi`** and **`s5`** (a blood measurement) push
> progression up the most. An $R^2$ around 0.4 says the ten linear features explain roughly 40% of
> the variance &mdash; modest, but honest, because it was measured on held-out patients. (For
> continuity, try re-running this section on the apartments data from Session&nbsp;01: `from nb_data
> import make_apartments`.)

## Recap & exercises

**Recap.**
* Score regressions with **MSE / RMSE / MAE / $R^2$** (all one-liners), always on **held-out** data.
* **Polynomial features** $[1, x, \dots, x^d]$ turn linear regression into curve fitting with the
  same machinery &mdash; "linear" means linear in the *parameters*.
* Raising the degree trades **bias for variance**: train error falls, test error makes a **U**, and
  the **weights explode** &mdash; the signature of overfitting.
* **Ridge** ($+\lambda\mathbf{I}$) shrinks the weights, tames the wiggle, and fixes singular
  $\mathbf{X}^{\!\top}\mathbf{X}$ &mdash; a one-line preview of Session&nbsp;03.
* The from-scratch fit matches scikit-learn on a **real** multivariate dataset.

**Exercises.**
1. Add MAPE (mean absolute percentage error) to the metrics and compute it in Section&nbsp;2. When
   is it undefined or misleading?
2. In Section&nbsp;3, standardize the polynomial columns before fitting (subtract mean, divide by
   std). Do the fitted curves change? Do the weight magnitudes?
3. In Section&nbsp;4, add a validation split and pick the degree on **validation** (per 01d), then
   report the test error. Does it match the U-curve's minimum?
4. Sweep ridge $\lambda \in \{0, 10^{-4}, 10^{-2}, 1, 100\}$ on the degree-15 fit. Plot the curves;
   which $\lambda$ best matches the true function? (This is Session&nbsp;03 in miniature.)
5. Re-run Section&nbsp;6 on the apartments data: one-hot encode `neighborhood` (per 01a), fit from
   scratch, and report test $R^2$. Which apartment features drive the rent?

*Next:* **Session&nbsp;03 &mdash; Regularization & model selection**, which turns the ridge one-liner
above into a full method (ridge, lasso, cross-validated $\lambda$) for controlling exactly the
overfitting we just watched happen.